In [748]:
# data pre processing

In [749]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import os
import joblib

In [750]:
df = pd.read_excel(r"C:\Users\yapen\Desktop\TrainDataset2025.xls")
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")

Total rows: 400
Total columns: 121


In [751]:
df = df.drop(columns=["pCR (outcome)", "ID"])

In [752]:
df = df.replace(999, np.nan)

print(f'Total Number of Missing Value is {df.isna().sum().sum()}')

print(f'Total Number of Row with Missing value is {(df.isna().any(axis=1)).sum()}')

print(f'Total Number of Column with Missing value is {(df.isna().any(axis=0)).sum()}')

Total Number of Missing Value is 100
Total Number of Row with Missing value is 90
Total Number of Column with Missing value is 8


In [753]:
mask = df.isna().sum(axis=1) > 1
print(f'Row with more than 1 Missing Value: \n{df.isna().sum(axis=1)[mask]}')

# delete Rows with more than 1 Missing Value
df = df.drop(df[mask].index)

Row with more than 1 Missing Value: 
225    3
261    4
267    3
294    4
dtype: int64


In [754]:
print('After Delete Rows\n')
print(f'Total Number of Missing Value is {df.isna().sum().sum()}')

print(f'Total Number of Row with Missing value is {(df.isna().any(axis=1)).sum()}')

print(f'Total Number of Column with Missing value is {(df.isna().any(axis=0)).sum()}')

After Delete Rows

Total Number of Missing Value is 86
Total Number of Row with Missing value is 86
Total Number of Column with Missing value is 2


In [755]:
print(f"The number of rows without LNStatus = {df['LNStatus'].isna().sum()}")
print(f"The number of rows without Gene = {df['Gene'].isna().sum()}")

The number of rows without LNStatus = 1
The number of rows without Gene = 85


In [756]:
df = df.dropna(subset=['LNStatus'])
print('Rows without LNStatus are Deleted')

Rows without LNStatus are Deleted


In [757]:
df = df.drop_duplicates()

In [758]:
print(df['Gene'].value_counts(dropna=False))
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)
df['Gene'] = imputer.fit_transform(df[['Gene']]).round().astype(int)

print(df['Gene'].value_counts(dropna=False))

Gene
0.0    192
1.0    118
NaN     85
Name: count, dtype: int64
Gene
0    277
1    118
Name: count, dtype: int64


In [759]:
X = df.drop(columns=['RelapseFreeSurvival (outcome)'])

y = df['RelapseFreeSurvival (outcome)']

print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")

Total rows: 395
Total columns: 119


In [760]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)



In [761]:
import numpy as np

threshold = 0.85  # you can change this

corr_matrix = X_train.corr().abs()


upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > threshold)]

print("High-correlated features dropped:", len(to_drop))
print(to_drop)

High-correlated features dropped: 74
['Proliferation', 'original_shape_Maximum2DDiameterSlice', 'original_shape_Maximum3DDiameter', 'original_shape_MinorAxisLength', 'original_shape_SurfaceArea', 'original_shape_VoxelVolume', 'original_firstorder_Energy', 'original_firstorder_Entropy', 'original_firstorder_MeanAbsoluteDeviation', 'original_firstorder_Mean', 'original_firstorder_Median', 'original_firstorder_Range', 'original_firstorder_RobustMeanAbsoluteDeviation', 'original_firstorder_RootMeanSquared', 'original_firstorder_TotalEnergy', 'original_firstorder_Uniformity', 'original_firstorder_Variance', 'original_glcm_ClusterProminence', 'original_glcm_ClusterShade', 'original_glcm_ClusterTendency', 'original_glcm_Contrast', 'original_glcm_DifferenceAverage', 'original_glcm_DifferenceEntropy', 'original_glcm_DifferenceVariance', 'original_glcm_Id', 'original_glcm_Idm', 'original_glcm_Idmn', 'original_glcm_Idn', 'original_glcm_Imc2', 'original_glcm_InverseVariance', 'original_glcm_JointA

In [762]:
X_train_fs = X_train.drop(columns=to_drop)

X_test_fs  = X_test.drop(columns=to_drop)


In [763]:
unnormalized_cols = []

for col in X_train_fs.columns:
    mean = X_train_fs[col].mean()
    std  = X_train_fs[col].std()
    minv = X_train_fs[col].min()
    maxv = X_train_fs[col].max()

    is_standardized = (abs(mean) < 0.1) and (abs(std - 1) < 0.1)
    is_normalized   = (minv >= -0.1) and (maxv <= 1.1)

    if not is_standardized and not is_normalized:
        unnormalized_cols.append(col)

print("Columns to scale:", len(unnormalized_cols))


Columns to scale: 27


In [764]:
unnormalized_indices = [X_train_fs.columns.get_loc(c) for c in unnormalized_cols]

from sklearn.preprocessing import StandardScaler
import numpy as np

scaler = StandardScaler()

# convert to numpy
X_train_np = X_train_fs.values

X_test_np  = X_test_fs.values

cols = unnormalized_indices

# fit scaler on train-only columns
scaler.fit(X_train_np[:, cols])

# copy arrays
X_train_std = X_train_np.copy()

X_test_std  = X_test_np.copy()

# transform only selected columns
X_train_std[:, cols] = scaler.transform(X_train_np[:, cols])

X_test_std[:, cols]  = scaler.transform(X_test_np[:, cols])


In [765]:
from sklearn.decomposition import PCA

# choose number of components
pca = PCA(n_components=0.95)   # keep 95% variance

# fit on train
pca.fit(X_train_std)

# transform all
X_train_pca = pca.transform(X_train_std)

X_test_pca  = pca.transform(X_test_std)

print("Original dim:", X_train_std.shape[1])
print("After PCA:", X_train_pca.shape[1])



Original dim: 44
After PCA: 18


In [766]:
from sklearn.preprocessing import StandardScaler


scale_y = True   # set False if you don’t want to scale y

if scale_y:
    y_scaler = StandardScaler()
    y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1,1))
    y_test_scaled  = y_scaler.transform(y_test.values.reshape(-1,1))
        # save the y-scaler
    
else:
    y_train_scaled = y_train.values.reshape(-1,1)
    y_test_scaled  = y_test.values.reshape(-1,1)

In [767]:
save_dir = "regression_data_Nov_20"
os.makedirs(save_dir, exist_ok=True)

joblib.dump(y_scaler, os.path.join(save_dir, "y_scaler.pkl"))

np.save(os.path.join(save_dir, "X_train.npy"), X_train_pca)

np.save(os.path.join(save_dir, "X_test.npy"),  X_test_pca)
np.save(os.path.join(save_dir, "y_train.npy"), y_train_scaled)

np.save(os.path.join(save_dir, "y_test.npy"),  y_test_scaled)